<a href="https://colab.research.google.com/github/speediedan/finetuning-scheduler/blob/main/src/fts_examples/notebooks/publish/gsm8k/fts_gsm8k_lightning.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" />
</a>

In [ ]:
# To run this notebook in a fresh environment (Colab, for example), uncomment the line below. The PyTorch API
# needs Fine-Tuning Scheduler 2.15, installed from main until 2.15 is released.
# %pip install 'finetuning-scheduler[examples] @ git+https://github.com/speediedan/finetuning-scheduler.git@main'


# Scheduled Fine-Tuning with the Fine-Tuning Scheduler Extension

![Fine-Tuning Scheduler logo](logo_fts.png){height="55px" width="401px"}

[Fine-Tuning Scheduler](https://finetuning-scheduler.readthedocs.io/en/stable/index.html) (FTS) accelerates and enhances
foundation model experimentation with flexible, multi-phase fine-tuning schedules. A schedule says which parameters to
thaw in each phase; FTS thaws them, adds them to the optimizer's parameter groups (optionally reinitializing the
optimizer or lr scheduler) and decides when each phase ends. Training with FTS is simple and brings several benefits:

- it conveniently abstracts and composes a broad space of fine-tuning methods
- expedites and facilitates exploration of model tuning dynamics
- it composes with interpretability research packages, expanding the space of experimental interventions

FTS runs a schedule through either of two interfaces, which share one scheduling core and one schedule format: the
[FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler)
callback for Lightning, which this notebook uses, and the PyTorch API's ``TuningScheduler`` for a training loop you write
yourself (experimental in 2.15; the
[PyTorch version of this tutorial](https://finetuning-scheduler.readthedocs.io/en/latest/examples/notebooks/gsm8k/fts_gsm8k_pytorch.html)
runs the same experiment with it).

Setup is straightforward, just install from PyPI! Since this notebook-based example requires a few additional packages (e.g.
``transformers``, ``datasets``), we installed the ``finetuning-scheduler`` package with the ``[examples]`` extra above.
Once the ``finetuning-scheduler`` package is installed, the [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) callback (FTS) is available for use with Lightning.
For additional installation options, please see the Fine-Tuning Scheduler [README](https://github.com/speediedan/finetuning-scheduler/blob/main/README.md).



<div style="display:inline">

Fundamentally, [Fine-Tuning Scheduler](https://finetuning-scheduler.readthedocs.io/en/stable/index.html) enables
scheduled, multi-phase, fine-tuning of foundation models. Gradual unfreezing (i.e. thawing) can help maximize
foundation model knowledge retention while allowing (typically upper layers of) the model to
optimally adapt to new tasks during transfer learning [1, 2, 3]

</div>

The [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) callback orchestrates the gradual unfreezing
of models via a fine-tuning schedule that is either implicitly generated (the default) or explicitly provided by the user
(more computationally efficient). Fine-tuning phase transitions are driven by
[FTSEarlyStopping](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSEarlyStopping)
criteria (a multi-phase extension of ``EarlyStopping`` packaged with FinetuningScheduler), user-specified epoch transitions or a composition of the two (the default mode).
A [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) training session completes when the
final phase of the schedule has its stopping criteria met. See
the [early stopping documentation](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.callbacks.EarlyStopping.html) for more details on that callback's configuration.

![FinetuningScheduler explicit loss animation](fts_explicit_loss_anim.gif){height="272px" width="376px"}


## Basic Usage

If no fine-tuning schedule is provided by the user, [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) will generate a
[default schedule](#The-Default-Fine-Tuning-Schedule) and proceed to fine-tune according to the generated schedule,
using default [FTSEarlyStopping](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSEarlyStopping) and [FTSCheckpoint](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSCheckpoint) callbacks with ``monitor=val_loss``.

```python
import lightning as L
from finetuning_scheduler import FinetuningScheduler
trainer = L.Trainer(callbacks=[FinetuningScheduler()])
```

## The Default Fine-Tuning Schedule

Schedule definition is facilitated via the [gen_ft_schedule](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.strategy_adapters.html#finetuning_scheduler.strategy_adapters.StrategyAdapter.gen_ft_schedule) method which dumps a default fine-tuning schedule (by default using a naive, 2-parameters per level heuristic) which can be adjusted as
desired by the user and/or subsequently passed to the callback. Using the default/implicitly generated schedule will likely be less computationally efficient than a user-defined fine-tuning schedule but is useful for exploring a model's fine-tuning behavior and can serve as a good baseline for subsequent explicit schedule refinement.
While the current version of [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) only supports single optimizer and (optional) lr_scheduler configurations, per-phase maximum learning rates can be set as demonstrated in the next section.

## Specifying a Fine-Tuning Schedule

To specify a fine-tuning schedule, it's convenient to first generate the default schedule and then alter the thawed/unfrozen parameter groups associated with each fine-tuning phase as desired. Fine-tuning phases are zero-indexed and executed in ascending order.

1. First, generate the default schedule to ``Trainer.log_dir``. It will be named after your
   ``LightningModule`` subclass with the suffix ``_ft_schedule.yaml``.

```python
    import lightning as L
    from finetuning_scheduler import FinetuningScheduler
    trainer = L.Trainer(callbacks=[FinetuningScheduler(gen_ft_sched_only=True)])
```

2. Alter the schedule as desired. Here, on the left, are the first phases of the schedule generated for this
   notebook's model, and on the right the schedule we tune it with below (highlighted: what the generated
   schedule does not have).

<table>
<tr><th>Generated: the first 3 of 155 phases</th><th>Explicit: <code>fts_explicit</code>'s 4 phases</th></tr>
<tr><td style="vertical-align:top"><pre>
0:
  params:
  - model.model.norm.weight
  - model.model.layers.27.post_attention_layernorm.weight
1:
  params:
  - model.model.layers.27.input_layernorm.weight
  - model.model.layers.27.mlp.down_proj.weight
2:
  params:
  - model.model.layers.27.mlp.up_proj.weight
  - model.model.layers.27.mlp.gate_proj.weight
...
</pre></td><td style="vertical-align:top"><pre>
0:
  params:
<mark>  - role:adapter # the LoRA adapters first</mark>
<mark>  max_transition_epoch: 3</mark>
1:
  params:
  - model\.model\.layers\.(2[4-7])\.(?!.*lora_).*
  - model\.model\.norm\.weight
<mark>  lr: 2.0e-05</mark>
2:
  params:
  - model\.model\.layers\.(1[6-9]|2[0-3])\.(?!.*lora_).*
3:
  params:
  - model\.model\.layers\.([0-9]|1[0-5])\.(?!.*lora_).*
</pre></td></tr>
</table>

3. Once the fine-tuning schedule has been altered as desired, pass it to
   [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) to commence scheduled training:

```python
import lightning as L
from finetuning_scheduler import FinetuningScheduler

trainer = L.Trainer(callbacks=[FinetuningScheduler(ft_schedule="/path/to/my/schedule/my_schedule.yaml")])
```

## Early-Stopping and Epoch-Driven Phase Transition Criteria


By default, [FTSEarlyStopping](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSEarlyStopping) and epoch-driven
transition criteria are composed. If a ``max_transition_epoch`` is specified for a given phase, the next fine-tuning phase will begin at that epoch unless [FTSEarlyStopping](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSEarlyStopping) criteria are met first.
If [FinetuningScheduler.epoch_transitions_only](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler.params.epoch_transitions_only) is ``True``, [FTSEarlyStopping](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSEarlyStopping) will not be used
and transitions will be exclusively epoch-driven.


<div class="alert alert-info">

**Tip:** Use of regex expressions can be convenient for specifying more complex schedules. Also, a per-phase base maximum lr can be specified:

<pre>
0:
<mark>  params: # the parameters for each phase definition can be fully specified</mark>
  - model.model.norm.weight
  - model.model.layers.27.post_attention_layernorm.weight
  max_transition_epoch: 3
1:
<mark>  params: # or specified via a regex</mark>
  - model\.model\.layers\.2[4-7]\.mlp\..*
2:
  params:
  - model\.model\.layers\.2[4-7]\.self_attn\.(?!.*lora_).*
  max_transition_epoch: 9
<mark>  lr: 1e-06 # per-phase maximum learning rates can be specified</mark>
3:
<mark>  params: # names, regexes and role selectors can be used in the same phase</mark>
  - model.model.layers.27.input_layernorm.weight
  - model\.model\.layers\.2[4-6]\.input_layernorm\.weight
  - role:adapter
</pre>

</div>



The end-to-end example in this notebook ([Scheduled Fine-Tuning for Math Reasoning](#Scheduled-Fine-Tuning-for-Math-Reasoning)) uses [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) in explicit mode to fine-tune a small language model on [GSM8K](https://huggingface.co/datasets/openai/gsm8k) math word problems, measuring what it retains as it goes.
Please see the [official Fine-Tuning Scheduler documentation](https://finetuning-scheduler.readthedocs.io/en/stable/index.html) if you are interested in a [CLI-based example](https://finetuning-scheduler.readthedocs.io/en/latest/examples/cli/superglue_cli.html) using the LightningCLI, which fine-tunes a model on the [RTE](https://huggingface.co/datasets/aps/super_glue) task of [SuperGLUE](https://super.gluebenchmark.com/).

## Resuming Scheduled Fine-Tuning Training Sessions

Resumption of scheduled fine-tuning training is identical to the continuation of
[other training sessions](https://lightning.ai/docs/pytorch/stable/common/trainer.html) with the caveat that the provided checkpoint must have been saved by a [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) session.
[FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) uses [FTSCheckpoint](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSCheckpoint) (an extension of ``ModelCheckpoint``) to maintain schedule state with special metadata.


```python
import lightning as L
from finetuning_scheduler import FinetuningScheduler
trainer = L.Trainer(callbacks=[FinetuningScheduler()])
trainer.ckpt_path="some/path/to/my_checkpoint.ckpt"
trainer.fit(...)
```

Training will resume at the depth/level of the provided checkpoint according to the specified schedule. Schedules can be altered between training sessions but schedule compatibility is left to the user for maximal flexibility. If executing a user-defined schedule, typically the same schedule should be provided for the original and resumed training sessions.

By default ([FinetuningScheduler.restore_best](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html?highlight=restore_best#finetuning_scheduler.fts.FinetuningScheduler.params.restore_best) is ``True``), [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) will attempt to restore the best available checkpoint before fine-tuning depth transitions.

```python
trainer = L.Trainer(callbacks=[FinetuningScheduler()])
trainer.ckpt_path="some/path/to/my_kth_best_checkpoint.ckpt"
trainer.fit(...)
```

Note that similar to the behavior of [ModelCheckpoint](https://lightning.ai/docs/pytorch/stable/api/lightning.pytorch.callbacks.ModelCheckpoint.html), when resuming training with a
different [FTSCheckpoint](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSCheckpoint) ``dirpath`` from the provided
checkpoint, the new training session's checkpoint state will be re-initialized at the resumption depth with the provided checkpoint being set as the best checkpoint.

<div class="alert alert-warning">

**Note:** Currently, [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) supports the following distributed strategy types:

- ``ddp`` (and aliases ``ddp_find_unused_parameters_false``, ``ddp_find_unused_parameters_true``, ``ddp_spawn``, ``ddp_fork``, ``ddp_notebook``)
- ``fsdp`` (and alias ``fsdp_cpu_offload``)
- **NEW** ``ModelParallelStrategy`` (enabling use of PyTorch's composable distributed (e.g. ``fully_shard``, ``checkpoint``) and Tensor Parallelism (TP) APIs)
  - [See this example](https://finetuning-scheduler.readthedocs.io/en/latest/distributed/model_parallel_scheduled_fine_tuning.html)

Custom or officially unsupported strategies can be used by setting [FinetuningScheduler.allow_untested](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html?highlight=allow_untested#finetuning_scheduler.fts.FinetuningScheduler.params.allow_untested) to ``True``.
Note that most currently unsupported strategies are so because they require varying degrees of modification to be compatible. For example, ``deepspeed`` will require a [StrategyAdapter](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.strategy_adapters.html#finetuning_scheduler.strategy_adapters.StrategyAdapter) to be written (similar to the one for ``FSDP``, [FSDPStrategyAdapter](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.strategy_adapters.html#finetuning_scheduler.strategy_adapters.FSDPStrategyAdapter)) before support can be added (PRs welcome!),
while ``tpu_spawn`` would require an override of the current broadcast method to include python objects.
</div>

## Scheduled Fine-Tuning for Math Reasoning

The following example demonstrates the use of [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) to fine-tune a small language model,
[Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B), on [GSM8K](https://huggingface.co/datasets/openai/gsm8k), a set
of grade-school math word problems with worked solutions. Iterative early-stopping will be applied according to a
user-specified schedule.

Fine-tuning teaches a model a task, and it can also make the model forget some of what it knew before. Gradual
unfreezing is one answer to that [1, 2, 3]: train the parameters closest to the output first, and reach the rest of
the model only when the task needs them. To see the trade, every run here measures two losses at the end of each epoch:

- ``val_loss``, the task loss: the model's loss on the worked solutions of held-out GSM8K problems;
- ``retention_loss``, the model's loss on ordinary encyclopedic text
  ([WikiText-103](https://huggingface.co/datasets/Salesforce/wikitext)) it was not tuned on. It starts at the
  pretrained model's value, and a rise means the model is losing some of its general language modeling.

The schedule also thaws a set of low-rank adapters ([LoRA](https://arxiv.org/abs/2106.09685)), which we add to the
model ourselves in a dozen lines, so the example also shows how a schedule addresses parameters by their *role*
rather than by their names. The [PyTorch version of this tutorial](https://finetuning-scheduler.readthedocs.io/en/latest/examples/notebooks/gsm8k/fts_gsm8k_pytorch.html) runs the same experiment in a
training loop of its own, logging the same TensorBoard metric names, and reviews the runs' training trajectory.

In [ ]:
# papermill parameters: shrink these for a quick smoke run
MODEL_NAME = "Qwen/Qwen3-0.6B"
MAX_EPOCHS = 16
TRAIN_SUBSET = 1000  # GSM8K training problems to tune on (of 7,473; the docs render uses 4000); None keeps them all
VAL_SUBSET = 128  # held-out GSM8K problems the task loss is measured on
RETENTION_WINDOWS = 64  # 512-token WikiText-103 windows the retention loss is measured on
BATCH_SIZE = 8
SEED = 42
PRECISION = "auto"  # "auto" uses "bf16-mixed" where the GPU supports it, else "16-mixed" (a Colab T4, for example)

In [ ]:
import warnings

# datasets and transformers import tqdm, which warns once that ipywidgets is missing (its text progress bars
# work without it), so filter that before the imports below
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
import gc
import math
import os
import warnings

import datasets
import lightning as L
import torch
from lightning.pytorch.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from lightning.pytorch.loggers.tensorboard import TensorBoardLogger
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers import logging as transformers_logging

import finetuning_scheduler as fts
from finetuning_scheduler import CanonicalView

transformers_logging.set_verbosity_error()
datasets.logging.set_verbosity_error()
datasets.logging.disable_progress_bar()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
for warnf in [r".*does not have many workers.*", r".*The number of training batches.*"]:
    warnings.filterwarnings("ignore", warnf)

if PRECISION == "auto":
    bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
    PRECISION = "bf16-mixed" if bf16 else "16-mixed" if torch.cuda.is_available() else "32-true"
print(f"Training with precision {PRECISION!r}")

### Data

Each GSM8K example becomes a prompt (``Question: ...`` followed by ``Answer:``) and the worked solution. The loss is
taken on the solution's tokens only: the prompt's labels are ``-100``, which the model's loss ignores. The retention
windows are 512-token stretches of WikiText-103's validation text, scored with the usual next-token loss. The
``LightningDataModule`` serves them as a second validation dataloader.

In [ ]:
MAX_LEN = 384  # tokens: longer examples (a few dozen of GSM8K's) are truncated


class GSM8KDataModule(L.LightningDataModule):
    """GSM8K problems to tune on and validate with, and WikiText-103 windows to measure retention with."""

    def __init__(self, model_name_or_path: str, train_subset: int | None, val_subset: int, retention_windows: int,
                 batch_size: int, seed: int):
        super().__init__()
        self.save_hyperparameters()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)

    def encode(self, example) -> dict[str, list[int]]:
        prompt = self.tokenizer(f"Question: {example['question']}\nAnswer: ")["input_ids"]
        answer = self.tokenizer(example["answer"] + self.tokenizer.eos_token)["input_ids"]
        return {"input_ids": (prompt + answer)[:MAX_LEN], "labels": ([-100] * len(prompt) + answer)[:MAX_LEN]}

    def collate(self, rows) -> dict[str, torch.Tensor]:
        """Right-pad a batch; padding is masked out of attention and of the loss."""
        n = max(len(r["input_ids"]) for r in rows)
        ids = torch.full((len(rows), n), self.tokenizer.pad_token_id)
        labels = torch.full((len(rows), n), -100)
        mask = torch.zeros((len(rows), n), dtype=torch.long)
        for i, r in enumerate(rows):
            k = len(r["input_ids"])
            ids[i, :k], labels[i, :k], mask[i, :k] = torch.tensor(r["input_ids"]), torch.tensor(r["labels"]), 1
        return {"input_ids": ids, "labels": labels, "attention_mask": mask}

    def setup(self, stage: str) -> None:
        hp = self.hparams
        gsm8k = datasets.load_dataset("openai/gsm8k", "main")
        train_split = gsm8k["train"].shuffle(seed=hp.seed)
        self.train_rows = [self.encode(e) for e in train_split.select(range(hp.train_subset or len(train_split)))]
        self.val_rows = [self.encode(e) for e in gsm8k["test"].select(range(hp.val_subset))]
        wiki = datasets.load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1", split="validation")
        wiki_ids = self.tokenizer("\n".join(t for t in wiki["text"][:4000] if t.strip()))["input_ids"]
        windows = [wiki_ids[i * 512:(i + 1) * 512] for i in range(hp.retention_windows)]
        self.retention_rows = [{"input_ids": w, "labels": w} for w in windows]

    def train_dataloader(self) -> DataLoader:
        # a seeded shuffle, made afresh for each run: every scenario sees the same order, whatever ran before it
        return DataLoader(self.train_rows, batch_size=self.hparams.batch_size, shuffle=True, collate_fn=self.collate,
                          generator=torch.Generator().manual_seed(self.hparams.seed))

    def val_dataloader(self) -> list[DataLoader]:
        return [DataLoader(rows, batch_size=self.hparams.batch_size, collate_fn=self.collate)
                for rows in (self.val_rows, self.retention_rows)]

### Adding LoRA Adapters

A LoRA adapter leaves a linear layer's weight $W$ frozen and learns a low-rank update beside it: the layer computes
$W x + \frac{\alpha}{r} B A x$, with $B$ of shape $d_{out} \times r$ and $A$ of shape $r \times d_{in}$ for a small rank
$r$. $B$ starts at zero, so the adapted model starts out identical to the pretrained one. We wrap the query and value
projections of every attention block, which adds about 2.3 million parameters to the model's 596 million.

The adapters' parameters are named like any others (``model.model.layers.0.self_attn.q_proj.lora_a``), so a schedule
could list them by regex. We give them a *role* instead, and a schedule phase selects them with ``role:adapter``. This is how
that entry becomes a set of parameters:

1. ``add_lora`` gives the model 112 adapter tensors, the ``lora_a`` and ``lora_b`` of the query and value projections
   in each of the 28 decoder blocks, and the ``LightningModule`` maps each of their names to the role ``adapter``.
2. A [CanonicalView](https://finetuning-scheduler.readthedocs.io/en/latest/api/finetuning_scheduler.core.views.html#finetuning_scheduler.core.views.CanonicalView) over the ``LightningModule`` carries that mapping, attached to it as ``tunable_view``, where
   ``FinetuningScheduler`` finds it. The view keeps the module's own parameter names, and every parameter it is not
   given a role for has the role ``base``.
3. When fitting starts, ``FinetuningScheduler`` compiles the schedule through the view. A ``params`` entry that starts
   with ``role:`` is replaced by every parameter whose role matches it: ``role:adapter`` matches the role ``adapter``
   and any named one (``adapter:<name>``), while ``role:adapter:<name>`` would match only the adapters given that
   name. Every other entry is a parameter name or a regular expression matched against the names. So ``role:adapter``
   compiles to exactly the 112 adapter tensors, and a model with more or fewer adapters, or adapters elsewhere, needs
   no change to the schedule.
4. A parameter can be thawed by only one phase: the scheduler refuses a schedule in which two phases claim the same
   parameter. That is why the block entries in the schedule below exclude the adapter tensors inside those blocks.

In [ ]:
LORA_RANK = 16


class LoRALinear(nn.Module):
    """A frozen linear layer plus a trainable low-rank update (``lora_a`` and ``lora_b``)."""

    def __init__(self, base: nn.Linear, rank: int = LORA_RANK, alpha: int = 2 * LORA_RANK):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.lora_a = nn.Parameter(torch.randn(rank, base.in_features) / math.sqrt(rank))
        self.lora_b = nn.Parameter(torch.zeros(base.out_features, rank))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x) + (x @ self.lora_a.t() @ self.lora_b.t()) * self.scale


def add_lora(model: nn.Module) -> None:
    """Wrap every attention block's query and value projections."""
    for layer in model.model.layers:
        for name in ("q_proj", "v_proj"):
            setattr(layer.self_attn, name, LoRALinear(getattr(layer.self_attn, name)))

### The LightningModule

The module logs the Lightning tutorial's metric names, which the PyTorch tutorial's runs log too: ``train_loss`` at
every optimizer step, ``val_loss`` and ``retention_loss`` (one per validation dataloader, without Lightning's
dataloader-index suffix) and ``finetuning_schedule_depth`` at the end of each epoch, plus the number of trainable
parameters. A ``LearningRateMonitor`` adds each parameter group's learning rate.

The model keeps float32 weights, and the ``Trainer``'s mixed precision runs its forward pass in bfloat16 where the GPU
supports it and in float16 otherwise (with a gradient scaler). Gradient checkpointing recomputes each block's
activations in the backward pass rather than storing them, which keeps the baseline (every decoder block
trainable) within a 16 GB GPU such as a Colab T4.

In [ ]:
class GSM8KModule(L.LightningModule):
    """A causal language model fine-tuned on GSM8K, optionally with LoRA adapters given the ``adapter`` role."""

    def __init__(self, model_name_or_path: str, lr: float, lora: bool = False):
        super().__init__()
        self.save_hyperparameters()
        # fp32 master weights for mixed precision: transformers would otherwise keep the checkpoint's stored dtype
        self.model = AutoModelForCausalLM.from_pretrained(model_name_or_path, dtype=torch.float32)
        if lora:
            add_lora(self.model)
        # the token embeddings stay frozen in every scenario (the schedule never thaws them)
        self.model.get_input_embeddings().weight.requires_grad_(False)
        self.model.gradient_checkpointing_enable()
        self.model.config.use_cache = False
        # `from_pretrained` returns the model in eval mode, and Lightning trains a module in the mode it finds it in
        self.model.train()
        roles = {n: "adapter" for n, _ in self.named_parameters() if ".lora_" in n}
        self.tunable_view = CanonicalView(self, roles=roles)

    @property
    def finetuningscheduler_callback(self) -> fts.FinetuningScheduler | None:
        fts_callback = [c for c in self.trainer.callbacks if isinstance(c, fts.FinetuningScheduler)]
        return fts_callback[0] if fts_callback else None

    def training_step(self, batch, batch_idx: int):
        loss = self.model(**batch).loss
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx: int, dataloader_idx: int = 0) -> None:
        name = ("val_loss", "retention_loss")[dataloader_idx]
        self.log(name, self.model(**batch).loss, add_dataloader_idx=False, prog_bar=True)

    def on_train_epoch_end(self) -> None:
        if self.finetuningscheduler_callback:
            self.log("finetuning_schedule_depth", float(self.finetuningscheduler_callback.curr_depth))
        self.log("trainable_params", float(sum(p.numel() for p in self.parameters() if p.requires_grad)))

    def configure_optimizers(self):
        # ``FinetuningScheduler`` ensures the optimizer configured here optimizes the parameters (and only those
        # parameters) scheduled for phase 0 of the current fine-tuning schedule
        params = [p for p in self.parameters() if p.requires_grad]
        return AdamW(params=params, lr=self.hparams.lr, weight_decay=0.0)

### Our Training Sessions

We'll be comparing two fine-tuning training configurations. Both depend upon a shared set of defaults, only differing
in their respective fine-tuning schedules.

| Experiment Tag    | Training Scenario Description                                                        |
|:-----------------:| ------------------------------------------------------------------------------------ |
| ``fts_explicit``  | Training with a fine-tuning schedule explicitly provided by the user                   |
| ``nofts_baseline``| A baseline fine-tuning training session (without scheduled fine-tuning or adapters)    |

A default schedule for Qwen3-0.6B (``FinetuningScheduler(gen_ft_sched_only=True)``) thaws two parameters per phase
from the last registered one, on down through 28 decoder blocks a pair at a time. That is useful for exploring a
model's tuning dynamics, but its phases are far too fine for a 28-block model, so ``fts_explicit`` groups them into
four phases instead, starting with the adapters:

0. the LoRA adapters (``role:adapter``) alone, for at most 3 epochs (``max_transition_epoch``), at the optimizer's
   learning rate;
1. the top 4 decoder blocks (24 to 27) and the final norm, with a phase learning rate of its own (``lr``);
2. the next 8 blocks down (16 to 23);
3. the remaining blocks (0 to 15).

Each phase adds to the ones before it, so the last phase trains the adapters, every decoder block and the final norm.
The ``nofts_baseline`` scenario trains every decoder block and the final norm from the start, in one phase; its model
has no adapters.

The token embeddings stay frozen throughout, in the baseline too (Qwen3-0.6B ties its output layer to them, and
their 156 million parameters' optimizer state would not fit beside the rest on a 16 GB GPU, such as the Colab
T4 these tutorials are sized to run on). Entries are regular
expressions, carrying the ``model.`` prefix of the ``LightningModule``'s attribute; ``(?!.*lora_)`` keeps the block
entries from also matching the adapters inside those blocks, so the adapters are thawed by their own phase. Phase 0
trains at the learning rate of the optimizer ``configure_optimizers`` returns (a schedule cannot set phase 0's
``lr``); phases after the first take their learning rate from the phase's ``lr`` or, without one, from
``base_max_lr``.

In [ ]:
# Please see the [FinetuningScheduler documentation](https://finetuning-scheduler.readthedocs.io/en/stable/index.html)
# for a full description of the schedule format
ft_schedule_yaml = r"""
0:
  params:
  - role:adapter
  max_transition_epoch: 3
1:
  params:
  - model\.model\.layers\.(2[4-7])\.(?!.*lora_).*
  - model\.model\.norm\.weight
  lr: 2.0e-05
2:
  params:
  - model\.model\.layers\.(1[6-9]|2[0-3])\.(?!.*lora_).*
3:
  params:
  - model\.model\.layers\.([0-9]|1[0-5])\.(?!.*lora_).*
"""
ft_schedule_name = "Qwen3_GSM8K_ft_schedule_explicit.yaml"
# Let's write the schedule to a file so we can simulate loading an explicitly defined fine-tuning schedule.
with open(ft_schedule_name, "w") as f:
    f.write(ft_schedule_yaml)

### Optimizer Configuration

Though other optimizers can arguably yield some marginal advantage contingent on the context,
the Adam optimizer (and the [AdamW version](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html) which
implements decoupled weight decay) remains robust to hyperparameter choices and is commonly used for fine-tuning
foundation language models. See (Sivaprasad et al., 2020) and (Mosbach, Andriushchenko & Klakow, 2020) for theoretical
and systematic empirical justifications of Adam and its use in fine-tuning large transformer-based language models.

The optimizer ``configure_optimizers`` returns starts with phase 0's parameters, so its learning rate is phase 0's:
``ADAPTER_LR`` for ``fts_explicit``, whose phase 0 is the adapters, and ``LR`` for the baseline. The top blocks' phase
sets ``LR`` as its own ``lr``, and ``base_max_lr`` sets the lower block phases' to half of it, so lower blocks move
more gently. The usual LoRA learning rate is about ten times the full fine-tuning one, but it made these adapters
forget the most in a short trial, which is why they use a lower one. The learning rate stays constant within a
phase. An lr scheduler returned by ``configure_optimizers``
composes with the schedule too, and can be reinitialized at phase transitions (see
[LR scheduler reinitialization](https://finetuning-scheduler.readthedocs.io/en/latest/advanced/lr_scheduler_reinitialization.html)).

In [ ]:
LR, ADAPTER_LR, LATER_PHASE_LR = 2e-5, 5e-5, 1e-5
PHASE0_LR = {"fts_explicit": ADAPTER_LR, "nofts_baseline": LR}  # the optimizer's learning rate is phase 0's

### Callback Configuration

The only callback required to invoke the [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) is the [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) callback itself.
Default versions of [FTSCheckpoint](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSCheckpoint) and [FTSEarlyStopping](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts_supporters.html#finetuning_scheduler.fts_supporters.FTSEarlyStopping)
(if not specifying ``epoch_transitions_only``) will be included ([as discussed above](#Basic-Usage)) if not provided
in the callbacks list. We configure all three below, as the PyTorch tutorial configures its transition and checkpoint
policies: a phase ends when ``val_loss`` has not improved by ``min_delta`` for ``patience`` epochs, and the best
checkpoint is restored at each phase boundary. Only the task loss drives the transitions: the retention loss is
measured, never optimized. The ``nofts_baseline`` scenario uses the core Lightning ``EarlyStopping`` and
``ModelCheckpoint`` callbacks with the same configuration.

In [ ]:
earlystopping_kwargs = {"monitor": "val_loss", "min_delta": 0.001, "patience": 2}
checkpoint_kwargs = {"monitor": "val_loss", "save_top_k": 1}
scenario_callbacks = {
    "fts_explicit": lambda: [
        fts.FinetuningScheduler(ft_schedule=ft_schedule_name, base_max_lr=LATER_PHASE_LR),
        fts.FTSEarlyStopping(**earlystopping_kwargs),
        fts.FTSCheckpoint(**checkpoint_kwargs),
    ],
    "nofts_baseline": lambda: [EarlyStopping(**earlystopping_kwargs), ModelCheckpoint(**checkpoint_kwargs)],
}

### Running the Scenarios

Each scenario seeds itself where its model is built and draws a freshly seeded shuffle of the training data, so no
scenario depends on the ones run before it. Before tuning, a ``validate`` call logs the pretrained model's
``val_loss`` and ``retention_loss`` at step 0, the starting point of both curves. ``fts_explicit``'s model carries the
adapters; the baseline makes it ordinary fine-tuning with early stopping, of every decoder block and the final norm at
once.

In [ ]:
# optionally start TensorBoard and watch the runs' curves while they train by uncommenting the next 2 lines
# %load_ext tensorboard
# %tensorboard --logdir lightning_logs
enable_progress_bar = False  # set to True to see each epoch's progress


def train(scenario: str) -> None:
    L.seed_everything(SEED)  # seed each scenario where its model is built, so none depends on the ones before it
    model = GSM8KModule(MODEL_NAME, lr=PHASE0_LR[scenario], lora=scenario == "fts_explicit")
    dm = GSM8KDataModule(MODEL_NAME, TRAIN_SUBSET, VAL_SUBSET, RETENTION_WINDOWS, BATCH_SIZE, SEED)
    logger = TensorBoardLogger("lightning_logs", name=scenario)
    common = {"precision": PRECISION, "accelerator": "auto", "devices": 1, "logger": logger,
              "enable_progress_bar": enable_progress_bar}
    L.Trainer(**common).validate(model, datamodule=dm, verbose=False)
    print(f"Beginning training the '{scenario}' scenario")
    trainer = L.Trainer(max_epochs=MAX_EPOCHS, callbacks=[*scenario_callbacks[scenario](), LearningRateMonitor()],
                        num_sanity_val_steps=0, **common)
    trainer.fit(model, datamodule=dm)
    print(f"{scenario}: finished after {trainer.current_epoch} epochs, best val_loss "
          f"{trainer.checkpoint_callback.best_model_score:.4f}")
    del model, trainer
    gc.collect()  # the module, trainer and callbacks reference each other
    if torch.cuda.is_available():
        print(f"{scenario}: peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()


for scenario in scenario_callbacks:
    train(scenario)

For a review of these scenarios' training trajectory (what each phase thaws, when it transitions, and how the task
and retention losses move), see the [PyTorch version of this tutorial](https://finetuning-scheduler.readthedocs.io/en/latest/examples/notebooks/gsm8k/fts_gsm8k_pytorch.html#reviewing-the-training-trajectory), which runs the same
scenarios.

Note that though this example is intended to capture a common usage scenario, substantial variation is expected
among use cases and models.
In summary, [FinetuningScheduler](https://finetuning-scheduler.readthedocs.io/en/stable/api/finetuning_scheduler.fts.html#finetuning_scheduler.fts.FinetuningScheduler) provides increased fine-tuning flexibility that can be useful in a variety of
contexts from exploring model tuning behavior to maximizing performance.

## Footnotes

- [Howard, J., & Ruder, S. (2018)](https://arxiv.org/pdf/1801.06146.pdf). Fine-tuned Language
 Models for Text Classification. ArXiv, abs/1801.06146. [↩](#Scheduled-Fine-Tuning-with-the-Fine-Tuning-Scheduler-Extension)
- [Chronopoulou, A., Baziotis, C., & Potamianos, A. (2019)](https://arxiv.org/pdf/1902.10547.pdf).
 An embarrassingly simple approach for transfer learning from pretrained language models. arXiv
 preprint arXiv:1902.10547. [↩](#Scheduled-Fine-Tuning-with-the-Fine-Tuning-Scheduler-Extension)
- [Peters, M. E., Ruder, S., & Smith, N. A. (2019)](https://arxiv.org/pdf/1903.05987.pdf). To tune or not to
 tune? adapting pretrained representations to diverse tasks. arXiv preprint arXiv:1903.05987. [↩](#Scheduled-Fine-Tuning-with-the-Fine-Tuning-Scheduler-Extension)
- [Sivaprasad, P. T., Mai, F., Vogels, T., Jaggi, M., & Fleuret, F. (2020)](https://arxiv.org/pdf/1910.11758.pdf).
 Optimizer benchmarking needs to account for hyperparameter tuning. In International Conference on Machine Learning
(pp. 9036-9045). PMLR. [↩](#Optimizer-Configuration)
- [Mosbach, M., Andriushchenko, M., & Klakow, D. (2020)](https://arxiv.org/pdf/2006.04884.pdf). On the stability of
fine-tuning bert: Misconceptions, explanations, and strong baselines. arXiv preprint arXiv:2006.04884. [↩](#Optimizer-Configuration)
- [Cobbe, K., Kosaraju, V., Bavarian, M., et al. (2021)](https://arxiv.org/pdf/2110.14168.pdf). Training verifiers to
solve math word problems. arXiv preprint arXiv:2110.14168. [↩](#Scheduled-Fine-Tuning-for-Math-Reasoning)
- [Hu, E. J., Shen, Y., Wallis, P., et al. (2021)](https://arxiv.org/pdf/2106.09685.pdf). LoRA: Low-rank adaptation of
large language models. arXiv preprint arXiv:2106.09685. [↩](#Adding-LoRA-Adapters)
